# Liquidity Stress Testing

In [1]:
import numpy as np
import pandas as pd

def model_risk_liquid(contract_data, hqla_0, stress_factor = 0.3):
    """ Calculates Gaps and LCR under stress"""
    df = contract_data.copy() # We do not want to modify the original data base
    df["Stressed_Inflow"] = df["Inflow"] * (1 - stress_factor) # We get less payments
    df["Stressed_Outflow"] = df["Outflow"] * (1 + stress_factor) # Increment in money exit

    # Gaps

    df["Marginal_Gap_Stressed"] = (df["Stressed_Inflow"] - df["Stressed_Outflow"])
    df["Cumulative_Gap_Stressed"] = df["Marginal_Gap_Stressed"].cumsum()

    # LCR

    flux_30 = df[df["Days"] <= 30] # We need 30 day flux for LCR
    total_out = flux_30["Stressed_Outflow"].sum()
    total_in = flux_30["Stressed_Inflow"].sum()
    LCR = hqla_0 / (total_out - min(total_in, 0.75 * total_out)) * 100

    # Do we have money left? Where is the horizon?
    
    df["Stressed_Available_Cash"] = hqla_0 + df["Cumulative_Gap_Stressed"]
    negative_m = df[df["Stressed_Available_Cash"] < 0]
    if negative_m.empty:
        survival_horiz = df["Days"].max()
    else: 
        survival_horiz = negative_m["Days"].iloc[0]
    return df, LCR, survival_horiz

In [2]:
data_balance = {
        "Days": [
            1,
            7,
            15,
            30,
            90,
        ],  # Maturity at 1 day, 1 week, etc
        "Inflow": [5.0, 12.0, 20.0, 45.0, 80.0],
        "Outflow": [8.0, 15.0, 25.0, 50.0, 60.0],
    }

df_contractual = pd.DataFrame(data_balance)

# HQLA
emergency = 50.0

# Ejecutamos el modelo con un estrés del 35% 
df_result, lcr_final, horiz = model_risk_liquid(
    df_contractual, hqla_0=emergency, stress_factor=0.35
)

print("=== MATRIX UNDER STRESS ===")
print(df_result.to_string(index=False))
print("\n=======================================")
print(f"LCR UNDER STRESS: {lcr_final:.2f}%")

# Validación regulatoria
if lcr_final >= 100:
    print("Result: Our HQLA resists the stress test")
else:
    print(
        "Result: Not enough HQLA -> -> -> RISK!!"
    )

print("Survival Horizon at", horiz)

=== MATRIX UNDER STRESS ===
 Days  Inflow  Outflow  Stressed_Inflow  Stressed_Outflow  Marginal_Gap_Stressed  Cumulative_Gap_Stressed  Stressed_Available_Cash
    1     5.0      8.0             3.25             10.80                  -7.55                    -7.55                    42.45
    7    12.0     15.0             7.80             20.25                 -12.45                   -20.00                    30.00
   15    20.0     25.0            13.00             33.75                 -20.75                   -40.75                     9.25
   30    45.0     50.0            29.25             67.50                 -38.25                   -79.00                   -29.00
   90    80.0     60.0            52.00             81.00                 -29.00                  -108.00                   -58.00

LCR UNDER STRESS: 63.29%
Result: Not enough HQLA -> -> -> RISK!!
Survival Horizon at 30


# DV01, Macaulay duration, Parallel and Non-Parallel shocks

In [3]:
import numpy as np
import pandas as pd


def analyze_interest_rate_risk(portfolio_df, yield_curve_df):
    """Calculates the PV, Macaulay duration, Modified duration and DV01 for a cash flow portfolio."""
    df = pd.merge(portfolio_df, yield_curve_df, on="Maturity_Years", how="left")

    # Present Value (PV) calculation
    df["PV"] = df["Cash_Flow"] / (
        (1 + df["Base_Rate"]) ** df["Maturity_Years"]
    )
    total_pv = df["PV"].sum()

    # Macaulay Duration (Temporal center of gravity)
    weighted_time = df["PV"] * df["Maturity_Years"]
    macaulay_dur = weighted_time.sum() / total_pv

    # DV01 -> numerical derivative calculating PV with a +1 basis point (+1 bp) shock
    df["Rate_1bp"] = df["Base_Rate"] + 0.0001
    df["PV_1bp"] = df["Cash_Flow"] / (
        (1 + df["Rate_1bp"]) ** df["Maturity_Years"]
    )

    # DV01 per bucket = (PV_base - PV_1bp) scaled to individual Euros
    df["DV01_Bucket_EUR"] = (df["PV"] - df["PV_1bp"]) * 1_000_000 # PV is in M of Euros
    total_dv01_eur = df["DV01_Bucket_EUR"].sum()  # Linearly additive

    # Modified duration derived from: DV01 = PV * Dmod * 0.0001
    portfolio_mod_duration = total_dv01_eur / (total_pv * 1_000_000 * 0.0001)

    buckets_report = df[
        [
            "Maturity_Years",
            "Cash_Flow",
            "Base_Rate",
            "PV",
            "DV01_Bucket_EUR",
        ]
    ].copy()

    summary = {
        "Total_PV_Millions": total_pv,
        "Macaulay_Duration_Years": macaulay_dur,
        "Modified_Duration": portfolio_mod_duration,
        "Total_DV01_EUR": total_dv01_eur,
    }

    return buckets_report, summary


def simulate_curve_scenarios(portfolio_df, yield_curve_df):
    """Simulates the portfolio P&L impact under parallel and non-parallel curve shocks

    (flattening/steepening scenarios).
    """
    _, base_summary = analyze_interest_rate_risk(portfolio_df, yield_curve_df)
    pv_base = base_summary["Total_PV_Millions"] # We calculate the initial PV

    # Scenario 1: Aggressive Parallel Shift (+150 basis points across all nodes)
    curve_parallel = yield_curve_df.copy()
    curve_parallel["Base_Rate"] = curve_parallel["Base_Rate"] + 0.0150
    _, summary_parallel = analyze_interest_rate_risk(  # We calculate the new PV with the Parallel shift
        portfolio_df, curve_parallel
    )  
    pnl_parallel = (summary_parallel["Total_PV_Millions"] - pv_base) * 1_000_000

    # Scenario 2: Non-Parallel Shift (Curve Flattening)
    # Short-term rates spike (+200 bp), mid-term increases (+100 bp), long-term remains unchanged (0 bp)
    curve_non_parallel = yield_curve_df.copy()

    # Shock vector per node: {1Y: +200bp, 2Y: +150bp, 3Y: +100bp, 5Y: +0bp}
    non_parallel_shocks = {1: 0.0200, 2: 0.0150, 3: 0.0100, 5: 0.0000}
    curve_non_parallel["Shock"] = curve_non_parallel["Maturity_Years"].map(
        non_parallel_shocks
    )
    curve_non_parallel["Base_Rate"] = (
        curve_non_parallel["Base_Rate"] + curve_non_parallel["Shock"]
    )

    _, summary_non_parallel = analyze_interest_rate_risk(
        portfolio_df, curve_non_parallel
    )
    pnl_non_parallel = (
        summary_non_parallel["Total_PV_Millions"] - pv_base
    ) * 1_000_000

    return pnl_parallel, pnl_non_parallel


# Base Portfolio Data (Cash flows in Millions)
portfolio_data = {
    "Maturity_Years": [1, 2, 3, 5],
    "Cash_Flow": [10.0, 25.0, 20.0, 60.0],
}
portfolio = pd.DataFrame(portfolio_data)

# Market Base Yield Curve Data 
curve_data = {
    "Maturity_Years": [1, 2, 3, 5],
    "Base_Rate": [0.0350, 0.0365, 0.0380, 0.0400],  # 3.50%, 3.65%, etc.
}
yield_curve = pd.DataFrame(curve_data)

# Run Sensitivity Analysis
buckets_report, global_metrics = analyze_interest_rate_risk(
    portfolio, yield_curve
)

print("=========================================================")
print("      QUANTITATIVE RISK REPORT BY TIME BUCKETS           ")
print("=========================================================")
print(
    buckets_report.to_string(
        index=False,
        formatters={
            "PV": "{:.4f}".format,
            "DV01_Bucket_EUR": "{:.2f}".format,
        },
    )
)

print("\n=========================================================")
print("                  AGGREGATED METRICS                     ")
print("=========================================================")
print(f"Total Present Value (PV): {global_metrics['Total_PV_Millions']:.4f} M€")
print(f"Macaulay Duration:        {global_metrics['Macaulay_Duration_Years']:.2f} years")
print(f"Modified Duration:        {global_metrics['Modified_Duration']:.2f}")
print(f"Total Portfolio DV01:     {global_metrics['Total_DV01_EUR']:.2f} € (loss per +1bp shift)")

# 4. Run Stress Testing Scenarios
pnl_par, pnl_no_par = simulate_curve_scenarios(portfolio, yield_curve)

print("\n=========================================================")
print("            ADVERSE SCENARIO STRESS TESTING              ")
print("=========================================================")
print(f"P&L Impact - Parallel Shift (+150 bps): {pnl_par:,.2f} €")
print(f"P&L Impact - Non-Parallel (Flattening):  {pnl_no_par:,.2f} €")
print("=========================================================")

      QUANTITATIVE RISK REPORT BY TIME BUCKETS           
 Maturity_Years  Cash_Flow  Base_Rate      PV DV01_Bucket_EUR
              1       10.0     0.0350  9.6618          933.42
              2       25.0     0.0365 23.2703         4489.51
              3       20.0     0.0380 17.8829         5167.47
              5       60.0     0.0400 49.3156        23702.60

                  AGGREGATED METRICS                     
Total Present Value (PV): 100.1306 M€
Macaulay Duration:        3.56 years
Modified Duration:        3.42
Total Portfolio DV01:     34293.00 € (loss per +1bp shift)

            ADVERSE SCENARIO STRESS TESTING              
P&L Impact - Parallel Shift (+150 bps): -4,958,164.02 €
P&L Impact - Non-Parallel (Flattening):  -1,349,389.54 €


# The Multi-Entity / Multi-Currency Balance Sheet Simulator

In [4]:
import numpy as np
import pandas as pd

# Set seed for reproducibility of Monte Carlo simulations
np.random.seed(42)


def run_multi_currency_alm_simulation(
    initial_balance_df, initial_fx_rates, fx_volatilities, correlation_matrix, steps=3
):
    """
    Simulates the dynamic evolution of a multi-entity, multi-currency balance sheet.
    Applies behavioral decay rates to liabilities, generates correlated FX shocks via Monte Carlo,
    and triggers automated layered hedging rules based on risk thresholds.
    
    Parameters:
    - initial_balance_df: DataFrame with columns ['Entity', 'Currency', 'Assets_Millions', 'Liabilities_Millions', 'Decay_Rate']
    - initial_fx_rates: Dict {Currency: Rate_vs_EUR} (e.g., {'USD': 1.10})
    - fx_volatilities: Dict {Currency: Volatility_Decimal} (e.g., {'USD': 0.15})
    - correlation_matrix: NumPy Array representing asset class/currency correlations
    - steps: Integer, number of projection periods (e.g., months)
    """
    
    # Cholesky Decomposition to generate correlated random shocks
    cholesky_matrix = np.linalg.cholesky(correlation_matrix)
    currency_order = list(initial_fx_rates.keys()) # Order matching the correlation matrix
    
    simulation_records = []
    current_balance = initial_balance_df.copy()
    current_fx_rates = initial_fx_rates.copy()
     
    EXPOSURE_THRESHOLD_EUR = 5.0  # Max unhedged open position allowed per bucket
    HEDGE_RATIO = 0.80             # Hedge 80% of the excess exposure to optimize transaction costs
    
    for t in range(1, steps + 1):
        # Generate correlated FX shocks using Geometric Brownian Motion (GBM) 
        independent_shocks = np.random.normal(0, 1, len(currency_order))
        correlated_shocks = np.dot(cholesky_matrix, independent_shocks)
        
        simulated_fx = {}
        for i, ccy in enumerate(currency_order):
            if ccy == "EUR":
                simulated_fx[ccy] = 1.0  # EUR is our base reporting currency
            else:
                vol = fx_volatilities[ccy]
                # Apply stochastic shock to the FX rate
                simulated_fx[ccy] = current_fx_rates[ccy] * np.exp(-0.5 * (vol**2) + vol * correlated_shocks[i])
        
        # Process each balance sheet item dynamically
        for idx, row in current_balance.iterrows():
            entity = row["Entity"]
            ccy = row["Currency"]
            assets = row["Assets_Millions"]
            liabilities = row["Liabilities_Millions"]
            decay_rate = row["Decay_Rate"]
            
            # Behavioral Model Application: Exponential decay of non-maturing deposits (Liabilities)
            liabilities_after_decay = liabilities * (1 - decay_rate)
            
            # Calculate Net Open Position (NOP) in local currency BEFORE hedging
            nop_local_pre = assets - liabilities_after_decay
            nop_eur_pre = nop_local_pre / simulated_fx[ccy]
            
            # Automated Layered Hedging Rule Engine
            hedge_executed_local = 0.0
            hedge_triggered = "NO"
            
            if abs(nop_eur_pre) > EXPOSURE_THRESHOLD_EUR: # we do not want micro-trades
                hedge_triggered = "YES"
                # Calculate the excess exposure in EUR
                excess_eur = nop_eur_pre - EXPOSURE_THRESHOLD_EUR if nop_eur_pre > 0 else nop_eur_pre + EXPOSURE_THRESHOLD_EUR
                # Target hedge amount in EUR
                hedge_amount_eur = excess_eur * HEDGE_RATIO # Remember that our ratio is 80%
                # Convert hedge action back to local currency
                hedge_executed_local = hedge_amount_eur * simulated_fx[ccy]
            
            # Post-hedge structural metrics
            final_assets = assets - hedge_executed_local
            nop_local_post = final_assets - liabilities_after_decay
            nop_eur_post = nop_local_post / simulated_fx[ccy]
            
            # Log results for analysis
            simulation_records.append({
                "Time_Step": t,
                "Entity": entity,
                "Currency": ccy,
                "FX_Rate_vs_EUR": simulated_fx[ccy],
                "Pre_Hedge_NOP_EUR": nop_eur_pre,
                "Hedge_Triggered": hedge_triggered,
                "Hedge_Action_Local": hedge_executed_local,
                "Post_Hedge_NOP_EUR": nop_eur_post,
                "Final_Assets_Local": final_assets,
                "Final_Liabilities_Local": liabilities_after_decay
            })
            
            # Update the balance sheet state for the next time horizon iteration
            current_balance.at[idx, "Assets_Millions"] = final_assets
            current_balance.at[idx, "Liabilities_Millions"] = liabilities_after_decay
            
        # Update current FX rates to follow the path path for next step pathing
        current_fx_rates = simulated_fx.copy()
        
    return pd.DataFrame(simulation_records)


# Define Initial Multi-Entity / Multi-Currency Balance Sheet (Values in local currency millions)
initial_balance_sheet = pd.DataFrame({
    "Entity": ["Ebury_UK", "Ebury_UK", "Ebury_EU", "Ebury_EU"],
    "Currency": ["GBP", "USD", "EUR", "USD"],
    "Assets_Millions": [45.0, 32.0, 85.0, 15.0],
    "Liabilities_Millions": [40.0, 12.0, 80.0, 24.0],
    "Decay_Rate": [0.05, 0.08, 0.03, 0.07]  # Behavioral deposit outflow rate per period
})

# Define FX Market Risk Parameters (Base Reporting Currency: EUR)
# Mapping index: 0 = EUR, 1 = GBP, 2 = USD
fx_base_rates = {"EUR": 1.0, "GBP": 0.85, "USD": 1.12}
fx_vols = {"EUR": 0.0, "GBP": 0.10, "USD": 0.14}  # Annualized currency volatilities

# Macro Correlation Matrix Setup [EUR, GBP, USD]
market_correlation = np.array([
    [1.0, 0.4, 0.3],  # EUR
    [0.4, 1.0, 0.5],  # GBP
    [0.3, 0.5, 1.0]   # USD
])

# Execute ALM Risk Simulation Run 
simulation_report = run_multi_currency_alm_simulation(
    initial_balance_sheet, fx_base_rates, fx_vols, market_correlation, steps=3
)

print("=========================================================================================")
print("                  QUANT ALM ENGINE: MULTI-ENTITY BALANCE SHEET SIMULATION                ")
print("=========================================================================================")

formatted_report = simulation_report.copy()
print(formatted_report.to_string(
    index=False,
    columns=["Time_Step", "Entity", "Currency", "FX_Rate_vs_EUR", "Pre_Hedge_NOP_EUR", "Hedge_Triggered", "Hedge_Action_Local", "Post_Hedge_NOP_EUR"],
    formatters={
        "FX_Rate_vs_EUR": "{:.4f}".format,
        "Pre_Hedge_NOP_EUR": "{:,.2f} M€".format,
        "Hedge_Action_Local": "{:,.2f} M".format,
        "Post_Hedge_NOP_EUR": "{:,.2f} M€".format
    }
))
print("=========================================================================================")

                  QUANT ALM ENGINE: MULTI-ENTITY BALANCE SHEET SIMULATION                
 Time_Step   Entity Currency FX_Rate_vs_EUR Pre_Hedge_NOP_EUR Hedge_Triggered Hedge_Action_Local Post_Hedge_NOP_EUR
         1 Ebury_UK      GBP         0.8519           8.22 M€             YES             2.19 M            5.64 M€
         1 Ebury_UK      USD         1.2144          17.26 M€             YES            11.91 M            7.45 M€
         1 Ebury_EU      EUR         1.0000           7.40 M€             YES             1.92 M            5.48 M€
         1 Ebury_EU      USD         1.2144          -6.03 M€             YES            -1.00 M           -5.21 M€
         2 Ebury_UK      GBP         0.8817           7.61 M€             YES             1.84 M            5.52 M€
         2 Ebury_UK      USD         1.2296           8.08 M€             YES             3.03 M            5.62 M€
         2 Ebury_EU      EUR         1.0000           7.81 M€             YES             2.25 M  

# VaR99 Monte Carlo Simulator 

In [1]:
import numpy as np
import pandas as pd


np.random.seed(43) # Same seed for reproductibility

def calculate_mon_car_VaR99(portfolio_value, weights, vcv_matrix, num_simulations=10_000):
    """ Calculates the 1-day VaR at 99% confidence level using a Monte Carlo simulation
    for a multi-currency portfolio.
    """
    num_assets = len(weights)
    cholesky_ma = np.linalg.cholesky(vcv_matrix)
    random_shock = np.random.normal(0,1,(num_assets, num_simulations))
    correl_return = np.dot(cholesky_ma, random_shock).T
    portfolio_sim_returns = np.dot(correl_return, weights) # R_p = \sum_{i=1}^{n} w_i R_i = w_1 R_1 + w_2 R_2
    var_99_percentile = np.percentile(portfolio_sim_returns, 1) # Sort returns to find the 1st percentile (lowest 1% of returns)
    var_99_money = -var_99_percentile * portfolio_value # Loss amount is expressed as a positive number
    
    return var_99_money, portfolio_sim_returns
def run_var_backtest(actual_pnl_history, var_threshold):
    """
    Performs a historical backtest of the VaR model to count exceptions
    and classify the model according to Basel traffic light system.
    """
    exceptions = actual_pnl_history < -var_threshold
    num_exceptions = np.sum(exceptions)
    total_days = len(actual_pnl_history)
    if num_exceptions <= 4:
        zone = "GREEN (Model is accurate and robust)"
    elif num_exceptions <= 9:
        zone = "YELLOW (Model warning: check for calibration issues)"
    else:
        zone = "RED (Model failed: underestimating tail risk drastically)"
    
    return num_exceptions, total_days, zone

# Portfolio definition: 100 Million EUR distributed in GBP (40%) and USD (60%)
PORTFOLIO_VALUE = 100.0  # Millions
portfolio_weights = np.array([0.40, 0.60])

# Diagonals are variances, off-diagonals are covariances
daily_vcv = np.array([
    [0.000064, 0.000032],  # GBP asset variance & covariance -> covar / (volatility_1 * volat_2) approx 40% correlation
    [0.000032, 0.000100]   # USD covariance & asset variance
])

var_99_m, simulated_returns = calculate_mon_car_VaR99(
    PORTFOLIO_VALUE, portfolio_weights, daily_vcv, num_simulations=100_000
)

print("=========================================================")
print("         QUANT RISK ENGINE: MONTE CARLO VaR99            ")
print("=========================================================")
print(f"Portfolio Net Worth:             {PORTFOLIO_VALUE:.2f} M€")
print(f"Estimated 1-Day VaR99:           {var_99_m:.4f} M€")
print(f"Max expected loss with 99% conf: {var_99_m * 1_000_000:,.2f} €")

# Simulate 250 days of real historical trading P&L (Synthetic Test Data)
# We simulate daily real market movements with slightly fatter tails (Student's t-dist)
# to see how the backtest reacts to realistic market friction
synthetic_real_returns = np.random.standard_t(df=5, size=250) * 0.008 # scaled to a 0.8% day volatility
actual_daily_pnl = synthetic_real_returns * PORTFOLIO_VALUE

# Perform Backtesting Validation
exceptions_count, days_tested, basel_zone = run_var_backtest(actual_daily_pnl, var_99_m)

print("\n=========================================================")
print("               BASEL AMENDMENT BACKTESTING               ")
print("=========================================================")
print(f"Historical Days Tested:          {days_tested} days")
print(f"VaR Breaches (Exceptions):       {exceptions_count}")
print(f"Empirical Exception Rate:        {(exceptions_count / days_tested) * 100:.2f}% (Target: 1.00%)")
print(f"Basel Traffic Light Zone:        {basel_zone}")
print("=========================================================")

         QUANT RISK ENGINE: MONTE CARLO VaR99            
Portfolio Net Worth:             100.00 M€
Estimated 1-Day VaR99:           1.8429 M€
Max expected loss with 99% conf: 1,842,890.59 €

               BASEL AMENDMENT BACKTESTING               
Historical Days Tested:          250 days
VaR Breaches (Exceptions):       6
Empirical Exception Rate:        2.40% (Target: 1.00%)
Basel Traffic Light Zone:        YELLOW (Model warning: check for calibration issues)
